# Analyse Kiprix — Groupe SCP2
Notebook d'exploration des données scrapées depuis kiprix.com.

In [ ]:
from pathlib import Path
import pandas as pd
import re

data_path = Path('../data/raw/kiprix_gp.json')
if not data_path.exists():
    raise FileNotFoundError(f'Fichier introuvable: {data_path.resolve()}')

df = pd.read_json(data_path)
print(f'Produits chargés: {len(df)}')
df.head()

In [ ]:
def parse_diff(value):
    match = re.search(r'([+-]?\s*\d+[\d\s.,]*)\s*%', str(value))
    if not match:
        return None
    cleaned = match.group(1).replace(' ', '').replace('\u00a0', '').replace(',', '.')
    try:
        return float(cleaned)
    except ValueError:
        return None

df['difference_numeric'] = df['difference'].apply(parse_diff)
df['price_france_num'] = pd.to_numeric(
    df['price_france'].astype(str).str.replace('€', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
    errors='coerce'
)
df['price_dom_num'] = pd.to_numeric(
    df['price_dom'].astype(str).str.replace('€', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
    errors='coerce'
)

df[['difference_numeric', 'price_france_num', 'price_dom_num']].describe()

In [ ]:
summary = {
    'total_produits': int(len(df)),
    'moyenne_ecart_pct': float(df['difference_numeric'].dropna().mean()) if df['difference_numeric'].notna().any() else 0.0,
    'mediane_ecart_pct': float(df['difference_numeric'].dropna().median()) if df['difference_numeric'].notna().any() else 0.0,
    'max_ecart_pct': float(df['difference_numeric'].dropna().max()) if df['difference_numeric'].notna().any() else 0.0,
    'min_ecart_pct': float(df['difference_numeric'].dropna().min()) if df['difference_numeric'].notna().any() else 0.0,
}
summary

In [ ]:
top5_dom = df.dropna(subset=['price_dom_num']).nlargest(5, 'price_dom_num')[['name', 'territory', 'price_dom', 'difference']]
top5_dom

In [ ]:
# Analyse des prix unitaires (si disponibles dans le JSON)
if {'unit_price_france', 'unit_price_dom', 'unit_reference'}.issubset(df.columns):
    unit_df = df.dropna(subset=['unit_price_dom'])[[
        'name', 'territory', 'unit_reference', 'unit_price_france', 'unit_price_dom'
    ]].copy()
    print(f"Produits avec prix unitaire: {len(unit_df)}")
    display(unit_df.head(10))
    print('Moyenne prix unitaire France :', round(unit_df['unit_price_france'].dropna().mean(), 2))
    print('Moyenne prix unitaire DOM    :', round(unit_df['unit_price_dom'].dropna().mean(), 2))
else:
    print('Colonnes de prix unitaires absentes. Regénère le JSON avec le scraper mis à jour.')

In [ ]:
try:
    import matplotlib.pyplot as plt

    ax = df['difference_numeric'].dropna().plot(kind='hist', bins=20, title='Distribution des écarts de prix (%)', figsize=(8, 4))
    ax.set_xlabel('Écart (%)')
    plt.show()

    if 'territory' in df.columns:
        ax2 = df.groupby('territory')['difference_numeric'].mean().plot(kind='bar', title='Écart moyen par territoire', figsize=(6, 4))
        ax2.set_ylabel('Écart moyen (%)')
        plt.show()
except ModuleNotFoundError:
    print('matplotlib non installé. Installe-le avec: pip install matplotlib')